# 02 — Compute on encrypted data

## Goal

Open the SDK workspace in a separate, compute-only `HESession`; calculate encrypted sum, mean, and population variance; and save encrypted results without accessing plaintext or a secret key.

## Setup

Run `01_owner_encrypt.ipynb` through `OWNER_ENCRYPT=PASS` first and leave its kernel running. This notebook must use a different kernel process. Both notebooks use only the installed `he_sdk` package.

## Steps

### 1. Open the same public workspace

In [ ]:
import json
import os
from pathlib import Path

from he_sdk import HESession, SecretKeyUnavailableError
from he_sdk.artifacts import list_ciphertexts

WORKSPACE = Path(
    os.getenv("HE_SDK_WORKSPACE", Path.home() / "he-sdk-notebook-workspace")
).expanduser().resolve()

compute = HESession.open_workspace(WORKSPACE)
encrypted_input = compute.load(WORKSPACE, name="input")

print("workspace:", WORKSPACE)
print("backend:", compute.capabilities.backend)
print("input metadata:", encrypted_input.metadata)

### 2. Compute encrypted reductions and persist their ciphertexts

In [ ]:
encrypted_results = {
    "sum": compute.sum(encrypted_input),
    "mean": compute.mean(encrypted_input),
    "variance": compute.variance(encrypted_input),
}
for operation, encrypted_result in encrypted_results.items():
    compute.save(encrypted_result, WORKSPACE, name=operation)

print("saved ciphertexts:", list_ciphertexts(WORKSPACE))

## Checks

The SDK must reject decryption from this session. This is the important check—not merely a convention in notebook code.

In [ ]:
try:
    compute.decrypt(encrypted_input)
except SecretKeyUnavailableError as error:
    print("decrypt blocked:", error)
else:
    raise AssertionError("compute-only session unexpectedly decrypted input")

manifest = json.loads((WORKSPACE / "manifest.json").read_text())
assert manifest["contains_plaintext"] is False
assert manifest["contains_secret_key"] is False
assert {"input", "sum", "mean", "variance"} <= set(list_ciphertexts(WORKSPACE))
print("COMPUTE_ONLY=PASS")

## Next steps

Close the compute session, return to the owner notebook, and run its result-decryption cell. This notebook learns the vector length and HE metadata required for evaluation, but never the input values or decrypted answers.

In [ ]:
compute.close()
print("compute session closed; return to notebook 01")